In [40]:
!pip install -q ultralytics

In [41]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

In [42]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [43]:
def get_centroids(frame):
  result = model.track(frame, persist=True, verbose=False)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  if result[0].boxes.id is None:
    return []
  ids = result[0].boxes.id.cpu().numpy()
  centroid=[]

  for box,c,tid in zip(boxes,conf,ids):
    if c < 0.5:
      continue
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy,tid))

  return centroid

In [44]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)

In [45]:
model = YOLO('yolov8n.pt')
players_position = {}
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
frame_count = 0
max_frames = 50

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)

    for cx, cy, tid in centroid:
        if tid in players_position:
            players_position[tid].append((cx, cy))
        else:
            players_position[tid] = [(cx, cy)]

print(players_position)

{np.float32(31.0): [(np.float32(403.42545), np.float32(304.4803)), (np.float32(403.37827), np.float32(304.47476)), (np.float32(403.25494), np.float32(304.3175)), (np.float32(403.2353), np.float32(304.37354)), (np.float32(403.00763), np.float32(304.43262)), (np.float32(402.7884), np.float32(304.25476)), (np.float32(402.71506), np.float32(304.18155)), (np.float32(402.59192), np.float32(304.26025)), (np.float32(402.50482), np.float32(304.28833)), (np.float32(402.3012), np.float32(304.2978)), (np.float32(402.25314), np.float32(304.13364)), (np.float32(402.22784), np.float32(304.01935)), (np.float32(402.1094), np.float32(303.8893)), (np.float32(402.10724), np.float32(304.10516)), (np.float32(402.04507), np.float32(304.35193)), (np.float32(401.86823), np.float32(304.39856)), (np.float32(401.4135), np.float32(304.4666)), (np.float32(401.03134), np.float32(304.42087)), (np.float32(400.35736), np.float32(304.27875)), (np.float32(399.69968), np.float32(303.49225)), (np.float32(399.3828), np.floa

In [46]:
players_position = {pid: pos for pid, pos in players_position.items() if len(pos) >= 5}

In [47]:
player_distances = {}

for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total

print(player_distances)

{np.float32(31.0): np.float32(48.62578), np.float32(32.0): np.float32(105.0585), np.float32(33.0): np.float32(49.13876), np.float32(34.0): np.float32(32.692703), np.float32(35.0): np.float32(34.77272), np.float32(36.0): np.float32(49.73112), np.float32(37.0): np.float32(59.196594), np.float32(38.0): np.float32(68.620544)}


In [48]:
def get_jersey_crop(frame, box):
    # cast to int since slicing needs whole numbers, not the floats YOLO gives
    x1, y1, x2, y2 = map(int, box)

    height = y2 - y1
    # only take top 35% of box height to isolate jersey, skip shorts/legs
    new_y2 = int(y1 + (0.35 * height))

    # rows (y) first, then columns (x) — standard image slicing order
    return frame[y1:new_y2, x1:x2]

In [49]:

def get_avg_color(crop):
  return crop.mean(axis = (0,1))

In [50]:
avg_colors = []
result = model.track(frame, persist=True, verbose=False)
boxes = result[0].boxes.xyxy.cpu().numpy()
if result[0].boxes.id is None:
    ids = []
else:
    ids = result[0].boxes.id.cpu().numpy()
for box, tid in zip(boxes, ids):
  crop = get_jersey_crop(frame, box)
  avg_color = get_avg_color(crop)
  avg_colors.append((tid, avg_color))

print(len(avg_colors))
print(avg_colors[0])

7
(np.float32(31.0), array([      106.5,      160.12,      141.41]))


In [51]:
data = np.array(avg_colors, dtype=np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

print(compactness, labels, centers)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (7, 2) + inhomogeneous part.

In [ ]:
player_team = {}
for i,box in enumerate(boxes):
  x1, y1, x2, y2 = box
  cx = (x1+x2)/2
  cy= (y1+y2)/2

  best_id = None
  best_distance = float("inf")
  for pid,pos in players_position.items():
    d = math.dist((cx,cy), pos[-1])
    if d<best_distance:
      best_distance = d
      best_id = pid

  if best_id not in player_team :
    player_team[best_id] = labels[i]

print(player_team)

In [ ]:
team_distance = {}
for pid, dist in player_distances.items():
  if pid not in player_team:
    continue
  team = int(player_team[pid])
  if team not in team_distance:
    team_distance[team]= 0
  team_distance[team] += dist

print(team_distance)


In [ ]:
import math
def get_player_speeds(position_history,fps):
  speeds =[]
  for i in range(1,len(position_history)):
    prev_point = position_history[i-1]
    current_point = position_history[i]
    x1, y1 = prev_point
    x2, y2 = current_point
    distance = math.sqrt((x2-x1)**2 + (y2-y1)**2)
    speed = distance*fps
    speeds.append(speed)
  return speeds



fps = cap.get(cv2.CAP_PROP_FPS)

# Pass a valid player ID (e.g., 0)
speeds = get_player_speeds(players_position[0], fps)
print(speeds)

In [ ]:
def count_sprints(speeds, threshold):

  sprint_count = 0
  was_sprinting = False
  for i in speeds:
    is_sprinting = i > threshold
    if is_sprinting and not was_sprinting :
      sprint_count +=1
    was_sprinting = is_sprinting

  return sprint_count

counts = count_sprints(speeds, 30)
print(counts)

In [ ]:
def build_player_summary(players_position, player_team, player_distances, fps, sprint_threshold):
  player_summary = {}
  for player_id,pos_history in players_position.items():
    # loop over each tracked player to pull together their stats into one summary
    team = player_team.get(player_id, "unknown")
    if not isinstance(team, str):
      team = team.item()
    distance = player_distances[player_id]

    speed = get_player_speeds(pos_history,fps)
    sprint_count = count_sprints(speed,sprint_threshold)

    player_summary[player_id] = {
        "team" : team,
        "distance" : distance,
        "speed" : speed,
        "sprint_count" : sprint_count
    }

  return player_summary


build_player_summary(players_position, player_team, player_distances, fps, 30)
